In [ ]:
import numpy as np
import matplotlib.pyplot as plt 
import pandas as pd 
import os

from ripser import ripser
from persim import plot_diagrams
from gtda.diagrams import PersistenceEntropy, PairwiseDistance
from gtda.diagrams import Silhouette
from gtda.diagrams import BettiCurve 
from gtda.time_series import TakensEmbedding, SlidingWindow, SingleTakensEmbedding, takens_embedding_optimal_parameters
import plotly.express as px
from gtda.homology import VietorisRipsPersistence
from gtda.metaestimators import CollectionTransformer
from gtda.pipeline import Pipeline
from sklearn.decomposition import PCA
from gtda.plotting import plot_point_cloud

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path('..').resolve()
sys.path.insert(0, str(REPO_ROOT / 'src'))

from tda_slugging.utils import read_files, find_shortest_file, batch_analyzer, mutual_information

# Data paths (type-3 slugging is committed; other types need the full 3W dataset)
DATA_3W_ALL      = str(REPO_ROOT / 'data' / '3W' / 'ALL')
DATA_3W_REAL     = str(REPO_ROOT / 'data' / '3W' / 'REAL')
DATA_3W_SIM      = str(REPO_ROOT / 'data' / '3W' / 'SIMULATED')
DATA_WELL        = str(REPO_ROOT / 'data' / 'well')
OUTPUTS_DIR      = str(REPO_ROOT / 'outputs')
IMAGES_DIR       = str(REPO_ROOT / 'images')
# Paths below require the full 3W dataset (download from https://github.com/ricardovvargas/3w_dataset):
# DATA_3W_NORMAL   = '/path/to/3w_dataset/data/0'
# DATA_3W_UNSTABLE = '/path/to/3w_dataset/data/4'


In [ ]:
x_timestamp = np.linspace(0, 10, 1000)
y_periodic = np.cos(x_timestamp)

n = 3
m = 2
y_N_periodic = np.cos(n * x_timestamp)
y_scaled = m * y_periodic 
y_shifted = m*n + y_periodic

In [ ]:
figShapes = px.line()
figShapes.add_scatter(x=(x_timestamp), y=(y_periodic), name="cos(x)")
#figShapes.add_scatter(x=(x_timestamp), y=(y_N_periodic), name="cos(n*x)")
figShapes.add_scatter(x=(x_timestamp), y=(y_scaled), name="m*cos(x)")
figShapes.show()

In [ ]:
stride = 1
signal = y_periodic

embedding_dimension = 3
embedding_time_delay = 10 #optimal_time_delay
embedder = SingleTakensEmbedding(
    parameters_type="fixed", n_jobs=-1, time_delay=embedding_time_delay, dimension=embedding_dimension, stride=stride
)

signal_embedded_cosx = embedder.fit_transform(signal)
fig = plot_point_cloud(signal_embedded_cosx)
fig

In [ ]:
def is_pos_def(x):
    return np.all(np.linalg.eigvals(x) >= 0)

print(signal_embedded_cosx.shape)
tmp = signal_embedded_cosx 
is_pos_def(signal_embedded_cosx[:3,:])

In [ ]:
np.linalg.eigvals(signal_embedded_cosx[:3,:])

In [ ]:
if not os.path.exists("images"):
    os.mkdir("images")
fig.write_image("images/fig1.svg")

In [ ]:
homology_dimensions = (0,1)
VRP = VietorisRipsPersistence(homology_dimensions=homology_dimensions)
PerDiagram_cosx = VRP.fit_transform_plot(signal_embedded_cosx[None,:,:])

In [ ]:
file1 = open("cosx_persdiag_H0.dat", "w")
file2 = open("cosx_persdiag_H1.dat", "w")
for i in PerDiagram_cosx[0,:]:
    if i[2] == 0:
        tmp = str(i[0]) + str(' ') + str(i[1]) + "\n"
        file1.writelines(tmp)
    elif i[2] == 1:
        tmp = str(i[0]) + str(' ') + str(i[1]) + "\n"
        file2.writelines(tmp)
file1.close()
file2.close()

In [ ]:
Betti = BettiCurve()
Betti_cosx = Betti.fit_transform_plot(PerDiagram_cosx)
Betti_cosx

In [ ]:
from random import gauss
from random import seed
# seed random number generator
seed(123456)
# create white noise series
white_noise_gen = [gauss(0.0, .50) for i in range(1000)]
white_noise = pd.Series(white_noise_gen)

stride = 2
signal = white_noise

embedding_dimension = 3 
embedding_time_delay = 100 #optimal_time_delay
embedder = SingleTakensEmbedding(
    parameters_type="fixed", n_jobs=-1, time_delay=embedding_time_delay, dimension=embedding_dimension, stride=stride
)

signal_embedded_wn = embedder.fit_transform(signal)
fig_noise = plot_point_cloud(signal_embedded_wn)
fig_noise

In [ ]:
PerDiagram_cosx = VRP.fit_transform_plot(signal_embedded_cosx[None,:,:])

In [ ]:
file1 = open("wn_persdiag_H0.dat", "w")
file2 = open("wn_persdiag_H1.dat", "w")
for i in PerDiagram_cosx[0,:]:
    if i[2] == 0:
        tmp = str(i[0]) + str(' ') + str(i[1]) + "\n"
        file1.writelines(tmp)
    elif i[2] == 1:
        tmp = str(i[0]) + str(' ') + str(i[1]) + "\n"
        file2.writelines(tmp)
file1.close()
file2.close()

In [ ]:
stride = 1
signal = y_shifted

embedding_dimension = 3 
embedding_time_delay = 10 #optimal_time_delay
embedder = SingleTakensEmbedding(
    parameters_type="fixed", n_jobs=-1, time_delay=embedding_time_delay, dimension=embedding_dimension, stride=stride
)

signal_embedded_shift = embedder.fit_transform(signal)
fig = plot_point_cloud(signal_embedded_shift)
fig

In [ ]:
stride = 1
signal = y_N_periodic

embedding_dimension = 3 
embedding_time_delay = 10 #optimal_time_delay
embedder = SingleTakensEmbedding(
    parameters_type="fixed", n_jobs=-1, time_delay=embedding_time_delay, dimension=embedding_dimension, stride=stride
)

signal_embedded_cos3x = embedder.fit_transform(signal)
plot_point_cloud(signal_embedded_cos3x)

In [ ]:
signal = y_scaled

embedding_dimension = 3 
embedding_time_delay = 10 #optimal_time_delay
embedder = SingleTakensEmbedding(
    parameters_type="fixed", n_jobs=-1, time_delay=embedding_time_delay, dimension=embedding_dimension, stride=stride
)

signal_embedded_2cosx = embedder.fit_transform(signal)
plot_point_cloud(signal_embedded_2cosx)

In [ ]:
homology_dimensions = (0,1)
VRP = VietorisRipsPersistence(homology_dimensions=homology_dimensions)
PerDiagram_cosx = VRP.fit_transform_plot(signal_embedded_cosx[None,:,:])

In [ ]:
PerDiagram_cosx

In [ ]:
PerDiagram_2cosx = VRP.fit_transform_plot(signal_embedded_2cosx[None,:,:])

In [ ]:
PerDiagram_cos3x = VRP.fit_transform_plot(signal_embedded_cos3x[None,:,:])

In [ ]:
Betti = BettiCurve(n_bins=100)
Betti_cosx = Betti.fit_transform_plot(PerDiagram_cosx)

In [ ]:
Betti.get_params()

In [ ]:
y_non_periodic_simple = np.cos(2*x_timestamp)*np.sin((x_timestamp*np.pi*1.5))

In [ ]:
from scipy import special
x = np.linspace(-25, 5, 501)
y_non_periodic, aip, bi, bip = special.airy(x)
y_non_periodic = y_non_periodic * np.sin(np.pi * x) * np.cos(2*x)

In [ ]:
figShapes = px.line()
figShapes.add_scatter(x=(x_timestamp), y=(y_non_periodic_simple), name="quasi-periodic")
figShapes.show()

In [ ]:
stride = 1
signal = y_non_periodic

from sklearn.decomposition import PCA

embedding_dimension = 5
embedding_time_delay = 25 #optimal_time_delay
embedder = SingleTakensEmbedding(
    parameters_type="search", n_jobs=-1, time_delay=embedding_time_delay, dimension=embedding_dimension, stride=stride
)

signal_embedded_cosx = embedder.fit_transform(signal)
pca = PCA(n_components=3)
pca_slugging = pca.fit_transform(signal_embedded_cosx)

plot_point_cloud(pca_slugging)

In [ ]:
x = np.linspace(0, 10, 501)
alpha = 1 / np.log(5)
beta = 1 / np.log(2)
y_non_periodic = np.pi*np.sin(2* np.pi * alpha * x) - np.cos(2* np.pi * beta * x)

In [ ]:
figShapes = px.line()
figShapes.add_scatter(x=(x_timestamp), y=(y_non_periodic_simple), name="quasi-periodic")
figShapes.show()

In [ ]:
stride = 1
signal = y_non_periodic

from sklearn.decomposition import PCA

embedding_dimension = 8
embedding_time_delay = 25 #optimal_time_delay
embedder = SingleTakensEmbedding(
    parameters_type="search", n_jobs=-1, time_delay=embedding_time_delay, dimension=embedding_dimension, stride=stride
)

signal_embedded_cosx = embedder.fit_transform(signal)
#print(embedding_dimension)
#if embedding_dimension > 3
pca = PCA(n_components=3)
pca_slugging = pca.fit_transform(signal_embedded_cosx)
#else:
  #  pca_slugging = signal_embedded_cosx
#pca_slugging = signal_embedded_cosx

plot_point_cloud(pca_slugging)

In [ ]:
if not os.path.exists("images"):
    os.mkdir("images")
fig.write_image("images/incomm.svg")

In [ ]:
PerDiagram = VRP.fit_transform_plot(signal_embedded_cosx[None,:,:])